# Exploración de datos 

Toca la talacha de revisar qué hay en los datos, antes de querer hacer cualquier cosa.

### Cargar datos y confirmar que lo hiciste bien

In [ ]:
# Confirmar cómo se descargaron los datos y que sean los que queremos 
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
print(sorted(p.name for p in RAW.iterdir()))   # para ver qué archivos descargó el script

['concentradohogar.csv']


In [ ]:
# Cargo los datos y cuido que no se pierdan los folios de vivienda y hogar
hogares = pd.read_csv(RAW / "concentradohogar.csv",
                      dtype={"folioviv": str, "foliohog": str})

In [9]:
hogares.describe()

,ubica_geo,tam_loc,est_socio,est_dis,upm,factor,clase_hog,sexo_jefe,edad_jefe,educa_jefe,...,mater_serv,material,servicio,deposito,prest_terc,pago_tarje,deudas,balance,otras_erog,smg
count,91414.000000,91414.000000,91414.000000,91414.000000,91414.000000,91414.000000,91414.000000,91414.000000,91414.000000,91414.000000,...,91414.000000,91414.000000,91414.000000,9.141400e+04,91414.000000,91414.000000,91414.000000,91414.000000,9.141400e+04,9.141400e+04
mean,15911.919148,2.526965,2.167830,320.454876,5419.840221,424.773339,2.121787,1.323386,51.537281,5.884132,...,437.440290,272.002960,165.437329,5.083510e+03,138.201119,1268.418394,480.077062,441.249934,3.913704e+02,2.240370e+04
std,9315.008029,1.335543,0.856087,201.082093,3032.579567,474.677171,0.643906,0.467771,15.994928,2.575050,...,5227.735138,3871.717469,2080.656565,2.793275e+04,1866.005384,7951.381442,3258.076472,5003.416681,7.937269e+03,7.275997e-12
min,1001.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000,1.000000,14.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2.240370e+04
25%,8019.000000,1.000000,2.000000,148.000000,2839.000000,141.000000,2.000000,1.000000,39.000000,4.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2.240370e+04
50%,15121.500000,3.000000,2.000000,308.000000,5433.000000,288.000000,2.000000,1.000000,51.000000,6.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2.240370e+04
75%,24028.000000,4.000000,3.000000,492.000000,8015.750000,526.000000,2.000000,2.000000,63.000000,8.000000,...,0.000000,0.000000,0.000000,2.950810e+03,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2.240370e+04
max,32056.000000,4.000000,4.000000,681.000000,10593.000000,7127.000000,5.000000,2.000000,106.000000,11.000000,...,989010.980000,870329.670000,185869.560000,4.677049e+06,244844.260000,543442.620000,293478.260000,751585.570000,1.076087e+06,2.240370e+04


Tenemos una base con 91,414 observaciones y 124 variables. 

### Folio vivienda - hogar (ojito con la ENIGH)

Resulta que hay una relación uno a muchos entre el folio de vivienda y de hogar. Una vivienda puede tener varios hogares, pero cada hogar pertenece a una sola vivienda. 

In [7]:
hogares_por_vivienda = hogares.groupby("folioviv").size()
hogares_por_vivienda.value_counts().sort_index()

1    89369
2      841
3       93
4       21
Name: count, dtype: int64

Vemos que hay viviendas con hasta 4 hogares, para no contar doble o descartar estos hogares, creamos un "id_hogar". De modo que cada fila tenga un id único y podamos identificar a cada uno de los hogares. Así que se crea uniendo los dos folios.

In [8]:
hogares["id_hogar"] = hogares["folioviv"] + "_" + hogares["foliohog"]

/tmp/ipykernel_5641/273330120.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hogares["id_hogar"] = hogares["folioviv"] + "_" + hogares["foliohog"]


### Reducir variables 

Dado que la base es grande y no usaremos todas las variables, una forma de eficientar la exploración será si nos quedamos con las variables importantes y que sabemos que podríamos necesitar.

In [ ]:
# Usando el word podemos hacer un primer filtro para quitar algunas variables 
# O podemos decidir con el catálgo en la página 215 

cols = ["folioviv", "foliohog", "factor", "upm", "est_dis", "ubica_geo", "tam_loc", "est_socio", "clase_hog", "sexo_jefe", "edad_jefe","educa_jefe", "tot_integ", "hombres", "mujeres", "menores", "p12_64","p65mas", "ocupados", "ing_cor", "ali_dentro", "salud", "ambul_serv", "aten_hosp", "medic_prod" ]
hogares = pd.read_csv(RAW / "concentradohogar.csv", usecols=cols,
                      dtype={"folioviv": str, "foliohog": str})

